In [156]:
import pandas as pd
from rapidfuzz import process, fuzz
import re

In [279]:
df1 = pd.read_csv("Region_və_məktəblərin_siyahısı_cleaned.csv")
df2 = pd.read_csv("schools_az_final.csv")


In [267]:
df1

,Bölgə,Məktəb
0,bakı şəhəri yasamal rayonu,13 saylı orta məktəb
1,bakı şəhəri yasamal rayonu,18 saylı orta məktəb
2,bakı şəhəri yasamal rayonu,20 saylı məktəb-lisey
3,bakı şəhəri yasamal rayonu,21 saylı orta məktəb
4,bakı şəhəri yasamal rayonu,31 saylı orta məktəb
...,...,...
4491,siyəzən rayonu,Gil-gilçay stansiya orta məktəbi (əvvəli Gil-g...
4492,siyəzən rayonu,Qozağacı kənd əsas məktəbi
4493,siyəzən rayonu,Yanıq Ələz kənd əsas məktəbi
4494,siyəzən rayonu,Daşlı Calğan kənd əsas məktəbi


In [217]:
# ===============================
# 1. df1 – Bölgəni təmizlə
# ===============================
def extract_region_df1(text):
    if not isinstance(text, str):
        return None
    text = text.lower()
    text = re.sub(r"(bakı şəhəri|naxçıvan mr|şəhəri|rayonu)", "", text)
    return text.strip()

df1["region_clean"] = df1["Bölgə"].apply(extract_region_df1)

# ===============================
# 2. Region siyahısı
# ===============================
regions = (
    df1["region_clean"]
    .dropna()
    .unique()
    .tolist()
)

# ===============================
# 3. df2 – Ünvan içindən region tap
# ===============================
def extract_region_df2(address):
    if not isinstance(address, str):
        return None
    address = address.lower()
    for r in regions:
        if re.search(rf"\b{re.escape(r)}\b", address):
            return r
    return None

df2["region_clean"] = df2["Ünvan"].apply(extract_region_df2)

# ===============================
# 4. Sütunları açıq ayır (ƏSAS DÜZƏLİŞ)
# ===============================
df1_pref = df1.add_prefix("df1_")
df2_pref = df2.add_prefix("df2_")

df1_pref["region_clean"] = df1["region_clean"]
df2_pref["region_clean"] = df2["region_clean"]
# ===============================
# 5. Merge
# ===============================
result = df1_pref.merge(
    df2_pref,
    how="left",
    on="region_clean"
)

# ===============================
# 6. Köməkçi sütunu sil
# ===============================
result = result.drop(columns=["region_clean"])


In [259]:
result.head()


,is_match,reason
0,False,blocked
1,False,low_score_68.75
2,False,low_score_68.75
3,False,low_score_24.13793103448276
4,False,low_score_29.629629629629633


In [ ]:
# Regionlara gprə qruplaşdırılmış məktəbləri match etmək üçün aşağıdakı kodu tətbiq edirik.
# Nəticə olaraq sadəcə 2103 sətirlik mümkün match-lər əldə edilir.
# =====================================================
# 1. Məktəb tipini müəyyən et
# =====================================================
def detect_school_type(text):
    if not isinstance(text, str):
        return None
    t = text.lower()

    if "peşə" in t:
        return "pese"
    if "lisey" in t:
        return "lisey"
    if "internat" in t:
        return "internat"
    if "musiqi" in t:
        return "musiqi"
    if "idman" in t:
        return "idman"
    if "orta məktəb" in t or "məktəb" in t:
        return "mekteb"

    return "digər"


# =====================================================
# 2. Məktəb nömrəsini çıxar
# =====================================================
def extract_school_number(text):
    if not isinstance(text, str):
        return None
    m = re.search(r"\b(\d{1,4})\b", text)
    return m.group(1) if m else None


# =====================================================
# 3. Məktəb adını normallaşdır
# =====================================================
def normalize_school(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(
        r"(tam orta|orta|ümumtəhsil|məktəb|lisey|gimnaziya|peşə|nömrəli|saylı|adına|respublika|xüsusi)",
        "",
        text
    )
    text = re.sub(r"\(.*?\)", "", text)  # mötərizədəkiləri sil
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# =====================================================
# 4. Feature engineering
# =====================================================
result["type1"] = result["Məktəb1"].apply(detect_school_type)
result["type2"] = result["Məktəb2"].apply(detect_school_type)

result["num1"] = result["Məktəb1"].apply(extract_school_number)
result["num2"] = result["Məktəb2"].apply(extract_school_number)

result["norm1"] = result["Məktəb1"].apply(normalize_school)
result["norm2"] = result["Məktəb2"].apply(normalize_school)


# =====================================================
# 5. Match qaydaları
# =====================================================
def is_valid_match(row):
    # Tip uyğunluğu əsas şərtdir
    same_type = row["type1"] == row["type2"]

    # Nömrə uyğunluğu
    num_match = row["num1"] and row["num2"] and row["num1"] == row["num2"]

    # Ad oxşarlığı
    name_score = fuzz.token_set_ratio(row["norm1"], row["norm2"])

    #Əsas qayda: eyni tip + nömrə
    if same_type and num_match:
        return True

    #Eyni tip + yüksək ad oxşarlığı
    if same_type and name_score >= 90:
        return True

    #İSTİSNA: korpus / nəzdində halları
    if same_type and num_match and (
        "korpus" in row["Məktəb2"].lower()
        or "nəzdində" in row["Məktəb1"].lower()
    ):
        return True

    #SON ÇIXIŞ YOLU: eyni nömrə + çox yüksək ad oxşarlığı
    if num_match and name_score >= 97:
        return True

    return False


# =====================================================
# 6. Match-ləri hesabla
# =====================================================
candidates = result[result.apply(is_valid_match, axis=1)].copy()

# =====================================================
# 7. ONE-to-ONE qaydası (əsas məktəb üzrə)
# =====================================================
candidates["score"] = candidates.apply(
    lambda r: (
        100 if (r["num1"] == r["num2"] and r["type1"] == r["type2"])
        else fuzz.token_set_ratio(r["norm1"], r["norm2"])
    ),
    axis=1
)

candidates = (
    candidates
    .sort_values("score", ascending=False)
    .drop_duplicates(subset=["Bölgə", "Məktəb1"], keep="first")
)

# =====================================================
# 8. Texniki sütunları sil
# =====================================================
matched_df = candidates.drop(
    columns=["type1", "type2", "num1", "num2", "norm1", "norm2", "score"]
)

matched_df


In [ ]:
matched_df.to_csv("matched.csv", index=False)

In [280]:

df1['_key1'] = (
    df1['Bölgə'].astype(str).str.strip().str.lower() + '||' +
    df1['Məktəb'].astype(str).str.strip().str.lower()
)

matched_df['_key2'] = (
    matched_df['Bölgə'].astype(str).str.strip().str.lower() + '||' +
    matched_df['Məktəb1'].astype(str).str.strip().str.lower()
)

In [281]:
final_df = df1.merge(
    matched_df,
    left_on='_key1',
    right_on='_key2',
    how='left',
    suffixes=('', '_matched')
)

In [274]:
final_df.head(10)

,Bölgə,Məktəb,_key1,Bölgə_matched,Məktəb1,Məktəb2,Ünvan,longitude,latitude,_key2
0,bakı şəhəri yasamal rayonu,13 saylı orta məktəb,bakı şəhəri yasamal rayonu||13 saylı orta məktəb,bakı şəhəri yasamal rayonu,13 saylı orta məktəb,Şəfiqə Əfəndizadə adına 13 nömrəli tam orta mə...,"Həsən bəy Zərdabi 215, Yasamal, Bakı, AZ1122",49.804882,40.391351,bakı şəhəri yasamal rayonu||13 saylı orta məktəb
1,bakı şəhəri yasamal rayonu,18 saylı orta məktəb,bakı şəhəri yasamal rayonu||18 saylı orta məktəb,bakı şəhəri yasamal rayonu,18 saylı orta məktəb,Mikayıl Müşfiq adına 18 nömrəli tam orta məktəb,"Şamil Əzizbəyov 153, Yasamal, Bakı, AZ1009",49.831839,40.378512,bakı şəhəri yasamal rayonu||18 saylı orta məktəb
2,bakı şəhəri yasamal rayonu,20 saylı məktəb-lisey,bakı şəhəri yasamal rayonu||20 saylı məktəb-lisey,bakı şəhəri yasamal rayonu,20 saylı məktəb-lisey,Arif Hüseynzadə adına 20 nömrəli məktəb-lisey ...,"Parlament 9, Yasamal, Bakı, AZ1009",49.818669,40.363664,bakı şəhəri yasamal rayonu||20 saylı məktəb-lisey
3,bakı şəhəri yasamal rayonu,21 saylı orta məktəb,bakı şəhəri yasamal rayonu||21 saylı orta məktəb,bakı şəhəri yasamal rayonu,21 saylı orta məktəb,Eldar Məmmədov adına 21 nömrəli tam orta məktəb,"Mirəli Seyidov 79, Yasamal, Bakı, AZ1100",49.806958,40.378541,bakı şəhəri yasamal rayonu||21 saylı orta məktəb
4,bakı şəhəri yasamal rayonu,31 saylı orta məktəb,bakı şəhəri yasamal rayonu||31 saylı orta məktəb,bakı şəhəri yasamal rayonu,31 saylı orta məktəb,Aslan Ağaverdiyev adına 31 nömrəli tam orta mə...,"Fuad İbrahimbəyov 17, Yasamal, Bakı, AZ1065",49.827590,40.382513,bakı şəhəri yasamal rayonu||31 saylı orta məktəb
5,bakı şəhəri yasamal rayonu,38 saylı orta məktəb,bakı şəhəri yasamal rayonu||38 saylı orta məktəb,bakı şəhəri yasamal rayonu,38 saylı orta məktəb,Aytəkin Məmmədov adına 38 nömrəli tam orta məktəb,"Böyükkişi Ağayev 101, Yasamal, Bakı, AZ1138",49.798756,40.390186,bakı şəhəri yasamal rayonu||38 saylı orta məktəb
6,bakı şəhəri yasamal rayonu,52 saylı orta məktəb,bakı şəhəri yasamal rayonu||52 saylı orta məktəb,bakı şəhəri yasamal rayonu,52 saylı orta məktəb,M.İ.Cuvarlinski adına 52 nömrəli tam orta məktəb,"Mirzə Cabbar Məmmədzadə 368, Yasamal, Bakı, AZ...",49.800521,40.395616,bakı şəhəri yasamal rayonu||52 saylı orta məktəb
7,bakı şəhəri yasamal rayonu,53 saylı orta məktəb,bakı şəhəri yasamal rayonu||53 saylı orta məktəb,bakı şəhəri yasamal rayonu,53 saylı orta məktəb,53 nömrəli tam orta məktəb,"Zahid Xəlilov 49, Yasamal, Bakı, AZ1141",49.811202,40.378026,bakı şəhəri yasamal rayonu||53 saylı orta məktəb
8,bakı şəhəri yasamal rayonu,60 saylı orta məktəb,bakı şəhəri yasamal rayonu||60 saylı orta məktəb,bakı şəhəri yasamal rayonu,60 saylı orta məktəb,Paşa Nəzərov adına 60 nömrəli tam orta məktəb,"Abbasqulu ağa Bakıxanov 5, Yasamal, Bakı, AZ1065",49.825916,40.387049,bakı şəhəri yasamal rayonu||60 saylı orta məktəb
9,bakı şəhəri yasamal rayonu,133 saylı orta məktəb,bakı şəhəri yasamal rayonu||133 saylı orta məktəb,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [282]:
final_df[final_df['Məktəb1'].notnull()]

,Bölgə,Məktəb,_key1,Bölgə_matched,Məktəb1,Məktəb2,Ünvan,longitude,latitude,_key2
0,bakı şəhəri yasamal rayonu,13 saylı orta məktəb,bakı şəhəri yasamal rayonu||13 saylı orta məktəb,bakı şəhəri yasamal rayonu,13 saylı orta məktəb,Şəfiqə Əfəndizadə adına 13 nömrəli tam orta mə...,"Həsən bəy Zərdabi 215, Yasamal, Bakı, AZ1122",49.804882,40.391351,bakı şəhəri yasamal rayonu||13 saylı orta məktəb
1,bakı şəhəri yasamal rayonu,18 saylı orta məktəb,bakı şəhəri yasamal rayonu||18 saylı orta məktəb,bakı şəhəri yasamal rayonu,18 saylı orta məktəb,Mikayıl Müşfiq adına 18 nömrəli tam orta məktəb,"Şamil Əzizbəyov 153, Yasamal, Bakı, AZ1009",49.831839,40.378512,bakı şəhəri yasamal rayonu||18 saylı orta məktəb
2,bakı şəhəri yasamal rayonu,20 saylı məktəb-lisey,bakı şəhəri yasamal rayonu||20 saylı məktəb-lisey,bakı şəhəri yasamal rayonu,20 saylı məktəb-lisey,Arif Hüseynzadə adına 20 nömrəli məktəb-lisey ...,"Parlament 9, Yasamal, Bakı, AZ1009",49.818669,40.363664,bakı şəhəri yasamal rayonu||20 saylı məktəb-lisey
3,bakı şəhəri yasamal rayonu,21 saylı orta məktəb,bakı şəhəri yasamal rayonu||21 saylı orta məktəb,bakı şəhəri yasamal rayonu,21 saylı orta məktəb,Eldar Məmmədov adına 21 nömrəli tam orta məktəb,"Mirəli Seyidov 79, Yasamal, Bakı, AZ1100",49.806958,40.378541,bakı şəhəri yasamal rayonu||21 saylı orta məktəb
4,bakı şəhəri yasamal rayonu,31 saylı orta məktəb,bakı şəhəri yasamal rayonu||31 saylı orta məktəb,bakı şəhəri yasamal rayonu,31 saylı orta məktəb,Aslan Ağaverdiyev adına 31 nömrəli tam orta mə...,"Fuad İbrahimbəyov 17, Yasamal, Bakı, AZ1065",49.827590,40.382513,bakı şəhəri yasamal rayonu||31 saylı orta məktəb
...,...,...,...,...,...,...,...,...,...,...
4485,siyəzən rayonu,Böyük Həmyə kənd orta məktəbi -S.Zeynalov adına,siyəzən rayonu||böyük həmyə kənd orta məktəbi ...,siyəzən rayonu,Böyük Həmyə kənd orta məktəbi -S.Zeynalov adına,Böyük Həmyə kənd orta məktəbi,"Məhəmmədhüseyn Şəhriyar, Siyəzən, AZ5313",49.143031,41.102142,siyəzən rayonu||böyük həmyə kənd orta məktəbi ...
4486,siyəzən rayonu,Balaca Həmyə kənd orta məktəbi,siyəzən rayonu||balaca həmyə kənd orta məktəbi,siyəzən rayonu,Balaca Həmyə kənd orta məktəbi,Balaca Həmyə kənd orta məktəbi,"Məhsəti Gəncəvi, Siyəzən, AZ5313",49.148026,41.072966,siyəzən rayonu||balaca həmyə kənd orta məktəbi
4490,siyəzən rayonu,Tağay kənd əsas məktəbi,siyəzən rayonu||tağay kənd əsas məktəbi,siyəzən rayonu,Tağay kənd əsas məktəbi,Tuğay kənd orta məktəbi,"Hüseyn Cavid, Siyəzən, AZ5312",49.131525,41.150975,siyəzən rayonu||tağay kənd əsas məktəbi
4491,siyəzən rayonu,Gil-gilçay stansiya orta məktəbi (əvvəli Gil-g...,siyəzən rayonu||gil-gilçay stansiya orta məktə...,siyəzən rayonu,Gil-gilçay stansiya orta məktəbi (əvvəli Gil-g...,Gilgilçay kənd orta məktəbi,"Abdulla Şaiq, Siyəzən, AZ5312",49.086807,41.140245,siyəzən rayonu||gil-gilçay stansiya orta məktə...


In [283]:
final_df.drop(columns=['_key1', 'Bölgə_matched', 'Məktəb1', '_key2'], inplace=True)

In [286]:
final_df.to_csv("Schools_with_coordinates.csv", index=False, encoding="utf-8-sig")